In [1]:
from helper_functions import import_flight_data, extract_wind_data
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
data_dir = '/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data'
wind_dir = '/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/TWI data'

df = import_flight_data(data_dir=data_dir,limit=1, recursive=True)


In [3]:
COLUMN_MAP = {
    # time
    "Year": "year",
    "QAR YEAR": "year",
    "Date: Year (Derived)": "year",

    "MONTH": "month",
    "QAR MONTH": "month",
    "Date (month)": "month",

    "DAY": "day",
    "QAR DAY": "day",
    "Date (day)": "day",

    "GMT - Hours (BCD)": "hour",
    "QAR HOUR": "hour",
    "UTC Hours": "hour",

    "GMT - Minutes (BCD)": "minute",
    "QAR MINUTE": "minute",
    "UTC Minutes": "minute",

    "GMT Seconds": "second",
    "QAR SECOND": "second",
    "UTC Seconds": "second",

    # altitude
    "Radio Altitude": "radio_altitude",
    "Radio Height 1": "radio_altitude",

    # vertical acceleration
    "Vertical Acceleration": "vert_acc",
    "Normal acceleration": "vert_acc",
}

In [4]:
N_arrivals = 1

arrival_data = import_flight_data(data_dir=data_dir, 
    limit=N_arrivals, add_source_file=True)#, usecols=['Time (secs)','MONTH', 'DAY', 'Year', 'GMT - Hours (BCD)', 'GMT - Minutes (BCD)', 'GMT Seconds', 'Vertical Acceleration', 'Radio Altitude'])

#arrival_data.columns.tolist()

In [5]:
N_arrivals_tot = 3140 - 70
N_arrivals = N_arrivals_tot

arrival_data = import_flight_data(data_dir=data_dir,
    limit=N_arrivals, add_source_file=True, usecols=['month', 'day', 'year', 'hour', 'minute', 'second', 'vert_acc', 'radio_altitude'])

In [7]:
full_wind_data = []
idx = 0
error_files = []

for flight_id, flight_df in arrival_data.groupby('source_file'):
    idx +=1
    print(f"Processing flight: {idx}/{len(arrival_data.groupby('source_file'))}")
    try:
        wind_df = extract_wind_data(flight_df, data_dir=wind_dir)
        wind_df['source_file'] = flight_id
        full_wind_data.append(wind_df)

    except ValueError as e:
        print(f"Error processing flight {flight_id}: {e}")
        error_files.append(flight_id)
full_wind_data = pd.concat(full_wind_data, ignore_index=True)
full_wind_data.to_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject.csv', index=False)

Processing flight: 1/2827
Processing flight: 2/2827
Processing flight: 3/2827
Processing flight: 4/2827
Processing flight: 5/2827
Processing flight: 6/2827
Processing flight: 7/2827
Processing flight: 8/2827
Processing flight: 9/2827
Processing flight: 10/2827
Processing flight: 11/2827
Processing flight: 12/2827
Processing flight: 13/2827
Processing flight: 14/2827
Processing flight: 15/2827
Processing flight: 16/2827
Processing flight: 17/2827
Processing flight: 18/2827
Processing flight: 19/2827
Processing flight: 20/2827
Processing flight: 21/2827
Processing flight: 22/2827
Processing flight: 23/2827
Processing flight: 24/2827
Processing flight: 25/2827
Processing flight: 26/2827
Processing flight: 27/2827
Processing flight: 28/2827
Processing flight: 29/2827
Processing flight: 30/2827
Processing flight: 31/2827
Processing flight: 32/2827
Processing flight: 33/2827
Processing flight: 34/2827
Processing flight: 35/2827
Processing flight: 36/2827
Processing flight: 37/2827
Processing

In [ ]:
summary_df.columns = [
    f"{col}_{stat}".replace('.', '_')
    for col, stat in summary_df.columns]

summary_df.to_csv("summary_2.csv", index=False)



In [1]:
import pandas as pd
import numpy as np
all_wind_data = pd.read_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject/All_wind_data_2.csv')

In [ ]:
def add_uv(df, prefix):
    speed = df[f"{prefix}.VectorMeanWindSpeed"]
    direction = df[f"{prefix}.VectorMeanWindDirection"]

    df[f"{prefix}_u"] = speed * np.cos(direction)
    df[f"{prefix}_v"] = speed * np.sin(direction)

    return df

all_wind_data = add_uv(all_wind_data, 'W04')
all_wind_data = add_uv(all_wind_data, 'W22')
half_wind_data = 

def slope(x):
    t = np.arange(len(x))
    return np.polyfit(t, x, 1)[0]


def p95(x):
    return np.percentile(x, 95)


def range_(x):
    return np.max(x) - np.min(x)


def mean_abs_change(x):
    return np.mean(np.abs(np.diff(x)))

def extract_features(group):
    feats = {}

    # --- vector stations ---
    for p in ["W04", "W22"]:
        for comp in ["u", "v"]:
            x = group[f"{p}_{comp}"].values

            feats[f"{p}_{comp}_mean"] = np.mean(x)
            feats[f"{p}_{comp}_std"] = np.std(x)
            feats[f"{p}_{comp}_p95"] = p95(x)
            feats[f"{p}_{comp}_range"] = range_(x)
            feats[f"{p}_{comp}_slope"] = slope(x)
            feats[f"{p}_{comp}_mac"] = mean_abs_change(x)

    # --- scalar wind speed features ---
    for p in ["W04", "W22"]:
        x = group[f"{p}.ScalarMeanWindSpeed"].values

        feats[f"{p}_speed_mean"] = np.mean(x)
        feats[f"{p}_speed_std"] = np.std(x)
        feats[f"{p}_speed_p95"] = p95(x)
        feats[f"{p}_speed_range"] = range_(x)
        feats[f"{p}_speed_slope"] = slope(x)

    # --- cross-station shear (IMPORTANT) ---
    w04_speed = group["W04.ScalarMeanWindSpeed"].values
    w22_speed = group["W22.ScalarMeanWindSpeed"].values
    diff = w04_speed - w22_speed
    feats["speed_shear_mean"] = np.mean(diff)
    feats["speed_shear_std"] = np.std(diff)
    feats["speed_shear_p95"] = p95(diff)
    feats["speed_shear_range"] = range_(diff)
    feats["speed_shear_slope"] = slope(diff)

    return pd.Series(feats)

feature_df = all_wind_data.groupby("source_file").apply(extract_features).reset_index()

/var/folders/yw/gbklf6sd5q3_sr89rbjcfv7w0000gn/T/ipykernel_5036/3427554038.py:66: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feature_df = all_wind_data.groupby("source_file").apply(extract_features).reset_index()


In [8]:
#feature_df.drop(columns=["source_file"], inplace=True)
feature_df.to_csv("feature_df_2.csv", index=False)